# Notebook 2 · Defining the objective: build an environment

**Time:** about 1½ hours.

**You'll learn**
- how to turn a problem into a Gymnasium environment: what the agent sees, what it can do, what counts
  as success and when an episode ends
- why the observation has to include velocity
- how to test a reward function before training anything

The environment is where you define the problem. In applied RL, getting it right is usually most of
the work.

> 📗 **This is the solutions copy.** The exercise version is `onboarding/02_build_an_environment.ipynb`. Try the exercises there first.

## The task: altitude hold

| | |
|---|---|
| Start | at rest, anywhere within 1,500 m of the 20,000 m target |
| Goal | get into the band 20,000 ± 250 m and stay there |
| Decisions | every 3 minutes: vent helium (0), do nothing (1) or drop ballast (2) |
| Flight | 40 decisions, so 2 hours |
| Ballast | enough for 20 drops |
| Lost | more than 3,000 m from the target ends the episode early |

Why decide every 3 minutes rather than every second? You saw in notebook 1 that an action takes a
minute or more to take full effect. Deciding much more often gives the agent lots of decisions whose
effects it can't see yet, which makes learning slower. After 3 minutes the balloon has settled at its
new speed. (The main project decides every minute, over a 12-hour flight.)

## Setup

`toy_env.py` holds the task's numbers, some running, scoring and plotting helpers, and the finished
environment (`AltitudeHoldEnv`) that notebooks 3 and 4 use. You're about to write your own version, so
try not to peek at it: we'll check yours against it at the end of Part 1.

In [ ]:
import sys
from pathlib import Path

# toy_env.py lives in onboarding/. This makes it importable from both the
# exercise copy (onboarding/) and the solutions copy (onboarding/solutions/).
here = Path.cwd()
if (here / "onboarding" / "toy_env.py").is_file():
    here = here / "onboarding"
sys.path.insert(0, str(here.parent if here.name == "solutions" else here))

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

from environments.core.balloon import Balloon
from environments.core.constants import BALLAST_DROP
from toy_env import (
    ALTITUDE_SCALE,
    BALLAST_DROPS,
    BAND,
    DECISION_SECONDS,
    DROP,
    FLIGHT_DECISIONS,
    LOST_DISTANCE,
    PHYSICS_DT,
    START_SPREAD,
    STAY,
    TARGET_ALTITUDE,
    VELOCITY_SCALE,
    VENT,
    AltitudeHoldEnv,
    check,
    default_reward,
    evaluate,
    plot_flights,
    run_episode,
)

## Part 1 · The environment

A Gymnasium environment is a class with three parts:

- `__init__` declares the `action_space` and `observation_space`, so that libraries know what to expect.
- `reset(seed)` starts a new episode and returns `(observation, info)`.
- `step(action)` moves the world on by one decision and returns
  `(observation, reward, terminated, truncated, info)`.

The observation is three numbers, each scaled to roughly −1 to 1. Neural networks learn best from
inputs of about that size, which matters in notebook 4.

| Index | Name | Meaning | Scaling |
|---|---|---|---|
| 0 | altitude error | altitude − target (positive means too high) | ÷ 3,000 m, clipped to ±1 |
| 1 | velocity | vertical speed (positive means rising) | ÷ 2 m/s, clipped to ±1 |
| 2 | ballast left | fraction of the ballast still on board | 0 to 1 |

The reward is passed in as a function `reward_fn(altitude_error, action)`, so that we can try different
ones in Part 3. For now we use `sparse_reward`: 1 for each decision inside the band, 0 outside.

### ✏️ Exercise 2.1 · Write `reset` and `step`

`__init__`, `_get_obs` and `_get_info` are done for you. Fill in:

**`reset`**
- Pick a start altitude uniformly within `START_SPREAD` of `TARGET_ALTITUDE` using
  `self.np_random.uniform(low, high)`. That's Gymnasium's random number generator: `super().reset(seed)`
  seeds it, so the same seed always gives the same flight.
- Create the balloon there: `Balloon(dim=1, ballast_initial=BALLAST_DROPS * BALLAST_DROP, position=np.array([start]))`.
  It works out how much helium makes it float at that altitude, so it starts at rest.
- Set `self.decisions` back to 0.

**`step`**
- Carry out the action: `DROP` calls `balloon.drop_ballast()`, but only if `balloon.is_ballast_empty`
  is false. `VENT` calls `balloon.vent_gas()`.
- Run the physics for one decision: `DECISION_SECONDS / PHYSICS_DT` calls of `balloon.update(PHYSICS_DT)`.
- Add one to `self.decisions`, then work out:
  - `terminated`: the balloon is more than `LOST_DISTANCE` from the target
  - `truncated`: the flight has used all `FLIGHT_DECISIONS`
  - `reward`: `self.reward_fn(altitude_error, action)`, or 0 if the balloon is lost
- Return `self._get_obs(), reward, terminated, truncated, self._get_info()`.

In [ ]:
def sparse_reward(altitude_error, action):
    """1 for every decision spent inside the band, 0 otherwise."""
    return 1.0 if abs(altitude_error) <= BAND else 0.0


class MyAltitudeHoldEnv(gym.Env):
    def __init__(self, reward_fn=sparse_reward):
        self.reward_fn = reward_fn
        self.action_space = spaces.Discrete(3)  # 0 vent, 1 stay, 2 drop
        self.observation_space = spaces.Box(
            low=np.array([-1.0, -1.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0], dtype=np.float32),
        )
        self.balloon = None
        self.decisions = 0

    def _get_obs(self):
        b = self.balloon
        return np.array(
            [
                np.clip((b.altitude - TARGET_ALTITUDE) / ALTITUDE_SCALE, -1.0, 1.0),
                np.clip(b.velocity / VELOCITY_SCALE, -1.0, 1.0),
                b.ballast_mass / (BALLAST_DROPS * BALLAST_DROP),
            ],
            dtype=np.float32,
        )

    def _get_info(self):
        return {"altitude": float(self.balloon.altitude), "velocity": float(self.balloon.velocity)}

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)  # seeds self.np_random
        # BEGIN SOLUTION
        start = TARGET_ALTITUDE + self.np_random.uniform(-START_SPREAD, START_SPREAD)
        self.balloon = Balloon(
            dim=1,
            ballast_initial=BALLAST_DROPS * BALLAST_DROP,
            position=np.array([start]),
        )
        self.decisions = 0
        # END SOLUTION
        return self._get_obs(), self._get_info()

    def step(self, action):
        # BEGIN SOLUTION
        b = self.balloon
        if action == DROP and not b.is_ballast_empty:
            b.drop_ballast()
        elif action == VENT:
            b.vent_gas()
        for _ in range(int(DECISION_SECONDS / PHYSICS_DT)):
            b.update(PHYSICS_DT)
        self.decisions += 1

        altitude_error = b.altitude - TARGET_ALTITUDE
        terminated = bool(abs(altitude_error) > LOST_DISTANCE)
        truncated = bool(self.decisions >= FLIGHT_DECISIONS)
        reward = 0.0 if terminated else float(self.reward_fn(altitude_error, action))
        return self._get_obs(), reward, terminated, truncated, self._get_info()
        # END SOLUTION

Gymnasium's `check_env` tests that an environment follows the rules: the right spaces, the right return
types, seeding that works. It raises an error explaining the problem if not. We skip its render check
because this environment doesn't draw anything.

A classic mistake it catches: `abs(x) > y` on NumPy numbers gives a NumPy bool, not a Python `bool`, and
Gymnasium insists on the latter. That's why `terminated` and `truncated` need wrapping in `bool(...)`.

In [ ]:
env = MyAltitudeHoldEnv()
check_env(env, skip_render_check=True)
check(True, "check_env found no problems")


def same_as_reference(seed, actions):
    """Run the same actions through your environment and toy_env's, and compare."""
    mine, reference = MyAltitudeHoldEnv(), AltitudeHoldEnv(reward_fn=sparse_reward)
    obs_mine, _ = mine.reset(seed=seed)
    obs_ref, _ = reference.reset(seed=seed)
    if not np.allclose(obs_mine, obs_ref):
        return False
    for action in actions:
        step_mine, step_ref = mine.step(action), reference.step(action)
        if not np.allclose(step_mine[0], step_ref[0]) or step_mine[1:4] != step_ref[1:4]:
            return False
        if step_mine[2] or step_mine[3]:
            break
    return True


rng = np.random.default_rng(0)
matches = all(same_as_reference(seed, rng.integers(0, 3, size=40)) for seed in range(5))
check(matches, "your environment behaves exactly like the reference in toy_env.py")

## Part 2 · A controller, and what the observation needs

Before any learning, let's write a controller by hand. It gives us a bar to measure learned agents
against, and it tests whether the observation holds what's needed to do the job.

`obs_for` below builds an observation from physical units, which makes controllers easy to test.

### ✏️ Exercise 2.2 · Altitude only

The obvious first try: drop ballast if the balloon is below the band, vent if it's above, and otherwise
do nothing. Remember that `obs[0]` is the altitude error divided by `ALTITUDE_SCALE`.

In [ ]:
def obs_for(altitude_error, velocity, ballast_left=1.0):
    """An observation built from metres and metres per second."""
    return np.array(
        [altitude_error / ALTITUDE_SCALE, velocity / VELOCITY_SCALE, ballast_left],
        dtype=np.float32,
    )


def altitude_only_controller(obs):
    # BEGIN SOLUTION
    altitude_error = obs[0] * ALTITUDE_SCALE
    if altitude_error < -BAND:
        return DROP
    if altitude_error > BAND:
        return VENT
    return STAY
    # END SOLUTION


check(altitude_only_controller(obs_for(-800, 0)) == DROP, "drops when 800 m too low")
check(altitude_only_controller(obs_for(800, 0)) == VENT, "vents when 800 m too high")
check(altitude_only_controller(obs_for(100, 0)) == STAY, "does nothing inside the band")

In [ ]:
flights = [run_episode(env, altitude_only_controller, seed) for seed in (2, 3, 4)]
plot_flights(flights, labels=["flight 1", "flight 2", "flight 3"], title="Altitude-only controller")
plt.show()
evaluate(altitude_only_controller, env)

It overshoots. By the time the balloon reaches the band it's moving quickly, so it sails straight
through, and the controller has to reverse it, again and again, spending ballast and helium each time.
It's the same problem as `lean_policy` in CartPole: the controller knows where the balloon *is*, but
not where it's *heading*.

The general rule is that the observation has to contain enough to predict what happens next. That's
called the **Markov property**. Altitude alone isn't enough here; altitude plus velocity nearly is.
It's why velocity is in our observation. It's also why the main project's observation includes the
wind at many altitudes, not just the balloon's own position.

### ✏️ Exercise 2.3 · Use the velocity

Write `velocity_controller`. A good approach: decide what velocity you *want*, heading for the target
and arriving in about 10 minutes (600 s) but never faster than 1 m/s. Then drop ballast if you're
rising too slowly, vent if you're rising too fast, and otherwise leave it alone:

```
wanted_velocity = clip(-altitude_error / 600, -1, 1)      # m/s
drop if velocity < wanted_velocity - 0.25
vent if velocity > wanted_velocity + 0.25
```

The 0.25 m/s of slack stops it reacting to every small error.

In [ ]:
def velocity_controller(obs):
    # BEGIN SOLUTION
    altitude_error = obs[0] * ALTITUDE_SCALE
    velocity = obs[1] * VELOCITY_SCALE
    wanted_velocity = float(np.clip(-altitude_error / 600.0, -1.0, 1.0))
    if velocity < wanted_velocity - 0.25:
        return DROP
    if velocity > wanted_velocity + 0.25:
        return VENT
    return STAY
    # END SOLUTION


flights = [run_episode(env, velocity_controller, seed) for seed in (2, 3, 4)]
plot_flights(flights, labels=["flight 1", "flight 2", "flight 3"], title="Velocity-aware controller")
plt.show()
score = evaluate(velocity_controller, env)
print(score)
check(score["in_band"] > 0.8, "spends over 80% of the time in the band")
check(score["actions"] < 10, "uses fewer than 10 actions per flight")

Much better: it gets into the band and stays there, with a handful of actions. This is the bar a
learning agent will have to clear. It's saved as `hold_controller` in `toy_env.py` for the next two
notebooks.

## Part 3 · Defining the objective

The reward is the only thing a learning agent tries to maximise. It doesn't know what you *meant*. If
the reward scores some silly behaviour above the behaviour you wanted, a good learner will find the
silly behaviour. So before training anything, test the reward: score a few simple policies with it and
check that the ranking makes sense.

Here's the line-up:

In [ ]:
rng = np.random.default_rng(0)


def do_nothing(obs):
    return STAY


def random_policy(obs):
    return int(rng.integers(3))


def fly_away(obs):
    return DROP  # drop ballast every time, until it runs out


policies = {
    "do nothing": do_nothing,
    "random": random_policy,
    "fly away": fly_away,
    "velocity controller": velocity_controller,
}

### ✏️ Exercise 2.4 · Four reward functions

`sparse_reward` is already written. Write the other three. Each takes `altitude_error` in metres and
the `action`:

- `distance_penalty`: minus the distance from the target in km, so `-abs(altitude_error) / 1000`.
- `fuel_penalty`: −0.1 for every action that isn't `STAY`, otherwise 0.
- `shaped_reward`: 1 inside the band; outside, it halves for every 250 m beyond the edge of the band,
  `0.5 ** ((distance - BAND) / 250)`; then subtract 0.1 if the action isn't `STAY`.

In [ ]:
def distance_penalty(altitude_error, action):
    # BEGIN SOLUTION
    return -abs(altitude_error) / 1000
    # END SOLUTION


def fuel_penalty(altitude_error, action):
    # BEGIN SOLUTION
    return -0.1 if action != STAY else 0.0
    # END SOLUTION


def shaped_reward(altitude_error, action):
    # BEGIN SOLUTION
    distance = abs(altitude_error)
    reward = 1.0 if distance <= BAND else 0.5 ** ((distance - BAND) / 250)
    if action != STAY:
        reward -= 0.1
    return reward
    # END SOLUTION


check(np.isclose(distance_penalty(-500, STAY), -0.5), "distance_penalty: -0.5 at 500 m away")
check(fuel_penalty(0, DROP) == -0.1 and fuel_penalty(0, STAY) == 0, "fuel_penalty: -0.1 per action")
check(
    all(
        np.isclose(shaped_reward(e, a), default_reward(e, a))
        for e in (-2000, -600, -250, 0, 120, 251, 900)
        for a in (VENT, STAY, DROP)
    ),
    "shaped_reward matches the reference",
)

Now score every policy under every reward. Each entry is the average return over the 50 evaluation
flights (this takes a few seconds):

In [ ]:
rewards = {
    "sparse": sparse_reward,
    "distance penalty": distance_penalty,
    "fuel penalty": fuel_penalty,
    "shaped": shaped_reward,
}
table = pd.DataFrame(
    {
        reward_name: {
            policy_name: evaluate(policy, MyAltitudeHoldEnv(reward_fn=reward_fn))["total_reward"]
            for policy_name, policy in policies.items()
        }
        for reward_name, reward_fn in rewards.items()
    }
).round(2)
table

**Q1.** Under `distance penalty`, why does *fly away* score better than *do nothing*?

<!-- BEGIN SOLUTION -->
Every decision costs reward, so the less time spent in the episode, the better. Flying away loses the
balloon within about 10 decisions, which ends the episode, so it collects far fewer penalties than a
balloon that sits several hundred metres from the target for all 40 decisions. An agent trained on this
reward would learn to get itself lost on purpose.

The lesson: if every reward is negative, ending the episode early looks attractive. Fix it by making
good states score positively, as `sparse` and `shaped` do, or by adding a big penalty for getting lost.
<!-- END SOLUTION -->

**Q2.** Under `fuel penalty`, what would an agent learn, and why?

<!-- BEGIN SOLUTION -->
To do nothing. The reward only says "don't spend fuel" and never mentions the target, so the best
possible score is 0, and doing nothing gets it. A reward that only penalises cost is satisfied by
inaction.
<!-- END SOLUTION -->

**Q3.** `sparse` and `shaped` both rank the policies sensibly. Why might `shaped` still be the better
choice? Think about what each one says about ballast and helium, and what an agent 1 km from the
target can learn from each.

<!-- BEGIN SOLUTION -->
Two reasons.

- **`sparse` never mentions resources.** An agent trained on it has no reason to save ballast or
  helium, so it will happily spend them on tiny corrections. `shaped` charges 0.1 per action, so the
  agent learns to act only when it's worth it.
- **`sparse` gives no signal far from the target.** Everywhere outside the band scores 0, so a balloon
  1 km away can't tell that moving closer is better than moving further away. It only learns once it
  stumbles into the band. In this small task that happens often enough (1 in 6 flights starts inside
  the band). In a bigger problem, like keeping within 10 km of a point somewhere in the whole sky, it
  can make learning very slow. `shaped` rises smoothly as the balloon gets closer, so every step
  carries information.

That's why `shaped` is the default in `toy_env.py`. It's also close to the main project's reward:
flat inside the station-keeping radius, decaying outside it, and scaled down by a cost for each action.
<!-- END SOLUTION -->

🚀 **Stretch**
- Write a controller that beats `velocity_controller` under the `shaped` reward. Try changing the
  600 s and the 0.25 m/s.
- Change `distance_penalty` so that flying away no longer pays. (Hint: the problem is in what happens
  when the balloon is lost.)

## Summary

- An environment defines the problem: its spaces, `reset` and `step`, and when episodes end.
- `check_env` catches interface mistakes early.
- The observation must hold enough to predict what happens next (the Markov property). For the balloon
  that means velocity, not just altitude.
- The reward *is* the objective. Test it on simple policies before training anything.

The finished environment is `AltitudeHoldEnv` in `toy_env.py`, with the shaped reward as its default.
**Next:** in notebook 3, an agent learns to fly it from scratch.

**Git checkpoint:** `git add onboarding/02_build_an_environment.ipynb`, commit and push.